In [78]:
import os
import json
import datetime
from dotenv import load_dotenv
from typing import TypedDict, List, Annotated, Dict, Literal
from datetime import datetime
from pydantic import BaseModel, Field
from operator import add
import shlex

from pathlib import Path
import base64
from icalendar import Calendar, Event
import pytz
from reader import Reader

from langchain_core.messages import HumanMessage, BaseMessage, AIMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain.agents import create_agent

from logger import Logger

from langgraph.graph import StateGraph
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import InMemorySaver

In [79]:
class TimeResult(BaseModel):
    """Схема ответа от Time агента"""
    schedule: dict[str, str] = Field(
        description="Расписание в формате {'год:месяц:число:час:минута-год:месяц:число:час:минута': 'событие'}, если чего то не хватает, пытайся вытащить по контексту, но все значения должны быть, дата сейчас: " + str(datetime.now())
    )
    action: str = Field(description='Краткое описание твоих действий, максимум 100 символов.')

class ImageResult(BaseModel):
    """Схема ответа от Time агента"""
    info: str = Field(description='то, что видешь на фото, полное описание.')
    action: str = Field(description='Краткое описание твоих действий, максимум 100 символов.')

class ValidationResult(BaseModel):
    """Схема ответа от Validation агента"""
    next: Literal['Time', 'END'] = Field(
        description="Решение: Time - доработать, END - все в норме"
    )
    reason: str = Field(
        description="Краткое обоснование решения"
    )
    action: str = Field(description='Краткое описание твоих действий, максимум 100 символов.')

class VisualizerResult(BaseModel):
    """Схема ответа от Time агента"""
    info: str = Field(description='Визуализация расписания, точь в точь как описано.')
    action: str = Field(description='Краткое описание твоих действий, максимум 100 символов.')
    next: Literal['Sender', 'Reporter'] = Field(description='Если надо кому то отправить письмо, то выбери Sender, если все ок, то выбери Reporter')
    mails: List[str] = Field(description='Почты или имена людей, к которым надо отправить сообщение, если есть.')

In [80]:
def send_mail(mails: List[str], text: StateGraph):
    '''отправляет сообщение по майлу'''
    Logger.write_log('tool send_mail', f'mail sended to {mails}', text)
    return ''

In [81]:
#------------------------------------------LOADING PROMPTS----------------------------
def open_prompt(file_path: str = 'prompts.json') -> dict:
    with open(file_path, 'r') as f:
        return json.load(f)
    
PROMPTS = open_prompt()

#--------------------------MODELS CREATION------------------------------
load_dotenv('api_keys.env')

GITHUB_PAT = os.getenv('GITHUB_PAT')
GEMINI_API_KEY1 = os.getenv('GEMINI_API_KEY1')
GEMINI_API_KEY2 = os.getenv('GEMINI_API_KEY2')

GITHUB_URL = os.getenv('GITHUB_URL')

GPT_MODEL = os.getenv('GPT_MODEL')
GEMINI_MODEL = os.getenv('GEMINI_MODEL')

main_model = ChatGoogleGenerativeAI(
    model=GEMINI_MODEL,
    api_key=GEMINI_API_KEY1,
    temperature=0.1,
    max_retries=2
)

# sub_model = ChatGoogleGenerativeAI(
#     model=GEMINI_MODEL,
#     api_key=GEMINI_API_KEY2,
#     max_retries=2
# )

sub_model = ChatOpenAI(
    model=GPT_MODEL,
    api_key=GITHUB_PAT,
    base_url=GITHUB_URL,
    max_retries=5
)

#------------------------------------------GRAPH LOGIC---------------------------------
class GraphState(TypedDict):
    schedule: dict[str, str]
    
    user_task: str
    val_query: str
    next_agent: str
    image: str
    
    content: str
    attempts: int
    bad: str
    mails: List[str]
    
Logger.init_log_file('logs.json')
    
#--------------------------------------CREATIN NODES-------------------------------
def Time(state: GraphState) -> GraphState:
    Logger.write_log('Time manager', 'вход')
    
    prompt = json.dumps(PROMPTS['Time'], ensure_ascii=False, indent=2)
    user_query = state['user_task']
    val_query = state.get('val_query', '')

    last_schedule = state.get('schedule', '')
    if last_schedule:
        last_schedule = "Добавь в это расписание все по запросу пользователя " + str(last_schedule)
    
    structured_model = main_model.with_structured_output(TimeResult)
    response = structured_model.invoke([HumanMessage([prompt, val_query, user_query, last_schedule])])
    
    schedule = response.schedule
    schedule = list(schedule.items())
    schedule.sort()
        
    Logger.write_log('Time manager', response.action, str(schedule))
    
    return {
        'schedule': dict(schedule)
    }
    
def IMG(state: GraphState) -> GraphState:
    Logger.write_log('Image analyzer', 'вход')
    
    prompt = json.dumps(PROMPTS['IMG'], ensure_ascii=False, indent=2)
    user_query = state['image']
    message = HumanMessage(content=[
            {"type": "text", "text": prompt},
            {
                "type": "image_url",
                "image_url": {"url": f"data:image/png;base64,{user_query}"}
            }
        ])
    
    structured_model = main_model.with_structured_output(ImageResult)
    response = structured_model.invoke([message])
    
    res = response.info
            
    Logger.write_log('Image analyzer', response.action, res)
    
    return {
        'user_task': state['user_task'] + res
    }
    
def VAL(state: GraphState) -> GraphState:
    Logger.write_log('Validation agent', 'вход')
    
    prompt = json.dumps(PROMPTS['VAL'], ensure_ascii=False, indent=2)
    user_query = state['user_task']
    schedule = state['schedule']
    structured_model = sub_model.with_structured_output(ValidationResult)
    
    response = structured_model.invoke([
        {"role": "system", "content": prompt},
        {"role": "user", "content": f"Расписание: {schedule}\nЗапрос пользователя: {user_query}"}
    ])
    
    next = response.next
    task = '' if next == 'END' else response.reason
    
    Logger.write_log('Validation agent', response.action, task)
    
    return {
        'val_query': task,
        'next_agent': next,
        'attempts': state.get('attempts', 0) + 1
    } 
    
def Visualizer(state: GraphState) -> GraphState:
    Logger.write_log('Visualizer', 'вход')
    
    prompt = json.dumps(PROMPTS['Visualizer'], ensure_ascii=False, indent=2)
    user_query = state['user_task']
    schedule = state['schedule']
    
    formatted_schedule = {}
    for key, value in schedule.items():
        parts = key.split(':')
        if len(parts) == 5:
            year, month, day, hour, minute = parts
            date_obj = datetime.date(int(year), int(month), int(day))
            weekday = date_obj.strftime('%A').upper()
            month_name = date_obj.strftime('%B').upper()
            readable_key = f"{weekday}, {int(day)} {month_name} {hour}:{minute}"
            formatted_schedule[readable_key] = value
        else:
            formatted_schedule[key] = value
    
    structured_model = sub_model.with_structured_output(VisualizerResult)
    response = structured_model.invoke([
        {"role": "system", "content": prompt},
        {"role": "user", "content": f"Расписание: {json.dumps(formatted_schedule, ensure_ascii=False)}\nЗапрос: {user_query}"}
    ])
    
    res = response.info
    
    Logger.write_log('Visualizer', response.action, res)
    
    return {
        'content': res,
        'mails': response.mails,
        'next_agent': response.next
    }

def Sender(state: GraphState) -> GraphState:
    Logger.write_log('Sender', 'вход')
    send_mail(state['mails'], state['content'])
    Logger.write_log('Sender', 'выход')

def Reporter(state: GraphState) -> GraphState:
    Logger.write_log('Reporter', 'вход')
    
    prompt = json.dumps(PROMPTS['Reporter'], ensure_ascii=False, indent=2)
    bad = state.get('bad', 'нет')
    user_query = state.get('user_task', '')
    schedule = state.get('schedule', {})
    
    logs = Logger.read_logs()
    agent_stats = Logger.get_agent_stats()
    
    context = {
        'total_actions': len(logs),
        'agents_stats': agent_stats,
        'recent_actions': logs[-5:] if len(logs) > 5 else logs,
        'schedule': schedule,
        'user_query': user_query,
        'bad_intent': bad
    }
    
    response = main_model.invoke([
        {"role": "system", "content": prompt},
        {"role": "user", "content": f"Данные для отчета: {json.dumps(context, ensure_ascii=False, indent=2)}"}
    ])
    
    try:
        report = response.content[0]['text']
    except:
        report = response.content
    
    Logger.write_log('Reporter', 'Отчет сгенерирован', report[:200] + '...')
    
    return {
        'report': report,
        'content': report
    }

#-----------------------------------CONDITIONAL EDGES LOGIC------------------------------
def ENTER_router(state: GraphState) -> str:
    img = state.get('image', None)
    if img:
        return 'IMG'
    return 'Time'
    
def VAL_router(state: GraphState) -> str:
    if state.get('attempts', 0) < 2: return state['next_agent']
    return 'END'

def Visualizer_router(state: GraphState) -> str:
    return state['next_agent']
    
#-------------------------------------------GRAPH CREATING-------------------------------------
memo = InMemorySaver()
workflow = StateGraph(GraphState)

workflow.add_node('IMG', IMG)
workflow.add_node('Time', Time)
workflow.add_node('VAL', VAL)
workflow.add_node('Visualizer', Visualizer)
workflow.add_node('Reporter', Reporter)
workflow.add_node('Sender', Sender)

workflow.set_conditional_entry_point(
    ENTER_router,
    {'IMG': 'IMG', 'Time': 'Time'}
)

workflow.add_edge('IMG', 'Time')
workflow.add_edge('Time', 'VAL')
workflow.add_edge('Sender', 'Reporter')

workflow.add_conditional_edges(
    'VAL',
    VAL_router,
    {'Time': 'Time', 'END': 'Visualizer'}
)

workflow.add_conditional_edges(
    'Visualizer',
    Visualizer_router,
    {'Reporter': 'Reporter', 'Sender': 'Sender'}
)

workflow.set_finish_point('Reporter')

app = workflow.compile(checkpointer=memo)
config = {"configurable": {"thread_id": "session_1"}}


In [82]:
def make_base_64(file_path: str) -> str:
    with open(file_path, 'rb') as f:
        return base64.b64encode(f.read()).decode('utf-8')
    
def create_ics_from_dict(events_dict: dict, filename: str = 'my_calendar.ics', zone: str = "Europe/Moscow"):
    cal = Calendar()
    cal.add('prodid', '-//My calendar//mxm.dk//')
    cal.add('version', '2.0')
    
    selected_timezone = pytz.timezone(zone)
    print(f"\nВыбран часовой пояс: {zone}")
    
    for time in events_dict:
        start, end = time.split('-')
        s_year, s_moth, s_day, s_hour, s_minute = map(int, start.split(':'))
        e_year, e_moth, e_day, e_hour, e_minute = map(int, end.split(':'))
        summary = events_dict[time]
        
        start = selected_timezone.localize(datetime(s_year, s_moth, s_day, s_hour, s_minute))
        end = selected_timezone.localize(datetime(e_year, e_moth, e_day, e_hour, e_minute))
        
        event = Event()
        event.add('summary', summary)
        event.add('dtstart', start)
        event.add('dtend', end)
        
        cal.add_component(event)
        
    with open(filename, 'wb') as f:
        f.write(cal.to_ical())
    
    print(f'📁 Файл создан: {os.path.abspath(filename)}')
    print('Перейдите по ссылке и импортируйте его: https://calendar.google.com/calendar/u/0/r/settings/export?pli=1')

image_extensions = ['.jpg', '.jpeg', '.png', '.webp', '.bmp', '.tiff', '.tif', '.avif', '.heic', '.jp2', '.dng']

while True:
    data = input("Введите сюда расписание, задачу и путь к файлу.")
    if not data:
        break
    
    try:
        parts = shlex.split(data) 
    except ValueError:
        parts = data.split()

    text_parts = []
    images = []
    for part in parts:
        clean_part = part.strip('"\'')
        if any(clean_part.lower().endswith(ext) for ext in image_extensions) and os.path.exists(clean_part):
            images.append(make_base_64(clean_part))
        elif Reader.is_supported(clean_part) and os.path.exists(clean_part):
            text_parts.append(Reader.read_document(clean_part))
        else:
            text_parts.append(part)

    bad = ''
    if '/' in data:
        bad = data.split('/')[1]
            
    user_input = ' '.join(text_parts)
    images = ' '.join(images)
            
    if images:
        response = app.invoke({
            'user_task': user_input,
            'image': images,
            'bad': bad
        }, config=config)
        
    else:
        response = app.invoke({
            'user_task': user_input,
            'bad': bad
        }, config=config)
        
    print(response['content'])
    
ans = input("Хотите получить рассылку на данный календарь? [Да/Нет]")
if not ans or ans.lower() == 'да':
    create_ics_from_dict(response['schedule'])

{'time': '2026-06-22 23:08:53', 'agent_name': 'Time manager', 'action': 'вход', 'dop_info': ''}
{'time': '2026-06-22 23:08:57', 'agent_name': 'Time manager', 'action': 'Создал ежедневное расписание тренировок по 2 часа до 23 июля.', 'dop_info': "[('2026:06:23:08:00-2026:06:23:10:00', 'Бег'), ('2026:06:24:08:00-2026:06:24:10:00', 'ОФП'), ('2026:..."}
{'time': '2026-06-22 23:08:57', 'agent_name': 'Validation agent', 'action': 'вход', 'dop_info': ''}
{'time': '2026-06-22 23:09:03', 'agent_name': 'Validation agent', 'action': 'Проверка расписания: занятия каждый день по 2 часа начинаются в 08:00 и заканчиваются в 10:00 — все корректно, нет повторяющихся занятий в одно и то же время для одной группы и нет нелогичных пересечений. Названия занятий различны и последовательность соответствует требованиям.', 'dop_info': ''}
{'time': '2026-06-22 23:09:03', 'agent_name': 'Visualizer', 'action': 'вход', 'dop_info': ''}


KeyboardInterrupt: 

In [ ]:
# В основном коде, после выполнения
def print_summary():
    """Выводит краткую сводку по логам"""
    logs = Logger.read_logs()
    stats = Logger.get_agent_stats()
    
    print("\n" + "="*50)
    print("📊 КРАТКАЯ СВОДКА")
    print("="*50)
    print(f"Всего действий: {len(logs)}")
    print("\nАгенты:")
    for agent, data in stats.items():
        print(f"  🤖 {agent}: {data['actions']} действий")
    print("="*50 + "\n")

# Использование после выполнения
print_summary()


📊 КРАТКАЯ СВОДКА
Всего действий: 8

Агенты:
  🤖 Time manager: 2 действий
  🤖 Validation agent: 2 действий
  🤖 Visualizer: 2 действий
  🤖 Reporter: 2 действий

